# 🔒 Détection des Requêtes SQL Malveillantes par Machine Learning

| Champ | Valeur |
|---|---|
| **Étudiante** | Oumaima Bimesmaren |
| **Filière** | Master BDSI — Big Data et Systèmes Intelligents |
| **Année universitaire** | 2025-2026 |
| **Module** | Cyber Sécurité |
| **Type de projet** | Projet académique ML appliqué à la cybersécurité |

---


## Partie 1 — Introduction

### 1.1 La cybersécurité

La **cybersécurité** regroupe l'ensemble des techniques, processus et pratiques visant à protéger les systèmes informatiques, les réseaux et les données contre les attaques, les accès non autorisés et les altérations. Avec l'essor des applications web connectées à des bases de données relationnelles, la sécurisation des échanges entre les utilisateurs et les serveurs est devenue un enjeu stratégique pour toute organisation.

### 1.2 Les attaques SQL Injection

L'**injection SQL** (*SQL Injection* ou *SQLi*) est l'une des vulnérabilités web les plus répandues et les plus dangereuses. Elle consiste à injecter du code SQL malveillant dans une requête envoyée à une base de données, en exploitant une validation insuffisante des entrées utilisateur.

Exemples de motifs d'attaque courants :

| Type d'injection | Exemple |
|---|---|
| Contournement d'authentification | `' OR '1'='1` |
| Extraction de données | `' UNION SELECT username, password FROM users --` |
| Destruction de données | `'; DROP TABLE users; --` |
| Injection aveugle temporelle | `' OR SLEEP(5) --` |

L'OWASP (*Open Web Application Security Project*) classe l'injection SQL parmi les **dix risques les plus critiques** pour la sécurité des applications web depuis de nombreuses années.

### 1.3 Les risques associés

- Vol massif de données personnelles, bancaires ou médicales
- Contournement des mécanismes d'authentification
- Altération ou suppression de bases de données entières
- Exécution de commandes système sur le serveur
- Sanctions réglementaires (RGPD) et atteinte à la réputation

### 1.4 Objectifs du projet

L'objectif est de développer un **système intelligent de classification binaire** capable d'analyser automatiquement une requête SQL et de la classer en :

- 🟢 **Classe 0** : Requête SQL **légitime**
- 🔴 **Classe 1** : Requête SQL **malveillante** (SQL Injection)

Le pipeline complet couvre : génération du dataset, prétraitement, feature engineering (TF-IDF), entraînement et évaluation de plusieurs algorithmes ML, puis déploiement d'une fonction de détection en temps réel.


## Partie 2 — Description du Dataset

Le dataset est constitué de deux colonnes :

| Colonne | Description |
|---|---|
| `query` | Texte brut de la requête SQL |
| `label` | `0` = légitime · `1` = malveillante (SQL Injection) |

Le code suivant génère le fichier `dataset/sql_injection_dataset.csv` contenant **1 200 échantillons équilibrés** (600 légitimes + 600 malveillantes) à partir de templates réalistes inspirés de l'OWASP.


In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import re
import random
import os

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import joblib

sns.set_style("whitegrid")
random.seed(42)
np.random.seed(42)

print("Bibliothèques importées avec succès.")


In [ ]:
# ---------------------------------------------------------------
# Génération du dataset : requêtes SQL légitimes et malveillantes
# ---------------------------------------------------------------

# Vocabulaire utilisé pour générer des variations réalistes
tables = ["users", "clients", "products", "orders", "employees", "invoices",
          "accounts", "students", "transactions", "logs", "sessions",
          "comments", "articles", "categories", "payments", "suppliers"]

columns = ["id", "name", "username", "email", "password", "price", "status",
           "created_at", "amount", "category", "phone", "address", "age",
           "role", "stock", "description"]

values = ["1", "2", "10", "100", "john", "alice", "2024-01-01", "active",
          "pending", "paris", "admin", "test", "50.5", "999", "guest", "client01"]


def gen_legit():
    """Génère une requête SQL légitime aléatoire."""
    t, t2 = random.choice(tables), random.choice(tables)
    c, c2 = random.choice(columns), random.choice(columns)
    v = random.choice(values)
    n = random.randint(1, 500)

    templates = [
        f"SELECT * FROM {t} WHERE id = {n}",
        f"SELECT {c}, {c2} FROM {t}",
        f"SELECT * FROM {t} WHERE {c} = '{v}'",
        f"SELECT COUNT(*) FROM {t}",
        f"SELECT * FROM {t} ORDER BY {c} DESC LIMIT {n}",
        f"SELECT * FROM {t} WHERE {c} LIKE '%{v}%'",
        f"INSERT INTO {t} ({c}, {c2}) VALUES ('{v}', '{values[0]}')",
        f"UPDATE {t} SET {c} = '{v}' WHERE id = {n}",
        f"DELETE FROM {t} WHERE id = {n}",
        f"SELECT * FROM {t} WHERE {c} BETWEEN {n} AND {n+50}",
        f"SELECT {c} FROM {t} GROUP BY {c}",
        f"SELECT * FROM {t} a JOIN {t2} b ON a.id = b.{t}_id",
        f"SELECT * FROM {t} WHERE {c} IS NOT NULL",
        f"SELECT DISTINCT {c} FROM {t}",
        f"SELECT * FROM {t} WHERE created_at > '2023-01-01'",
        f"SELECT AVG({c}) FROM {t}",
        f"SELECT * FROM {t} WHERE id IN ({n}, {n+1}, {n+2})",
        f"ALTER TABLE {t} ADD COLUMN {c2} VARCHAR(255)",
        f"SELECT * FROM {t} WHERE {c} = {n} AND {c2} = '{v}'",
        f"SELECT * FROM {t} LIMIT {n} OFFSET {n//2}",
    ]
    return random.choice(templates)


def gen_malicious():
    """Génère une requête SQL malveillante (motif d'injection classique et documenté)."""
    t = random.choice(tables)
    c = random.choice(columns)
    v = random.choice(values)
    n = random.randint(1, 500)

    templates = [
        "' OR '1'='1",
        "' OR 1=1 --",
        "admin' --",
        "admin' #",
        "' OR 'a'='a",
        "' OR 1=1#",
        "1' OR '1'='1' --",
        f"' UNION SELECT username, password FROM {t} --",
        "' UNION SELECT NULL, NULL, NULL --",
        f"'; DROP TABLE {t}; --",
        f"'; DELETE FROM {t} WHERE 1=1; --",
        f"' AND 1=0 UNION SELECT {c}, NULL FROM {t} --",
        f"' OR EXISTS(SELECT * FROM {t}) --",
        "' OR SLEEP(5) --",
        f"' OR (SELECT COUNT(*) FROM {t}) > 0 --",
        f"' AND SUBSTRING((SELECT {c} FROM {t} LIMIT 1),1,1)='a",
        f"1; DROP TABLE {t}",
        "' UNION SELECT table_name, NULL FROM information_schema.tables --",
        "' OR '1'='1' LIMIT 1 --",
        "x' AND email IS NULL; --",
        "' OR username LIKE '%admin%' --",
        "' WAITFOR DELAY '0:0:5' --",
        "' OR pg_sleep(5) --",
        "'; EXEC xp_cmdshell('dir'); --",
        f"' OR 1=1; SELECT * FROM {t}; --",
        "admin'/*",
        "' or 1=1 -- -",
        f"SELECT * FROM {t} WHERE id = {n} OR 1=1",
        f"SELECT * FROM {t} WHERE {c} = '' OR ''='",
        "' UNION ALL SELECT NULL,NULL,version() --",
        f"SELECT * FROM {t} WHERE username='{v}' OR '1'='1'",
        f"SELECT * FROM {t} WHERE username = '{v}' OR '1'='1' --",
        f"SELECT * FROM {t} WHERE {c} = '{v}' OR '1'='1'",
        f"SELECT * FROM {t} WHERE login='{v}' AND password='' OR '1'='1'",
        f"SELECT * FROM {t} WHERE {c}='{v}' OR 1=1 --",
        f"SELECT * FROM {t} WHERE username='{v}' OR 1=1#",
        f"SELECT * FROM {t} WHERE id={n} OR '1'='1' --",
        f"SELECT * FROM {t} WHERE {c}='{v}' UNION SELECT username, password FROM {t} --",
        f"SELECT * FROM {t} WHERE username = '{v}' AND password = '' OR ''=''",
    ]
    return random.choice(templates)


# Exemples de base fournis dans le cahier des charges
base_examples = [
    ("SELECT * FROM users WHERE id=1", 0),
    ("SELECT * FROM products", 0),
    ("admin' OR '1'='1", 1),
    ("UNION SELECT username,password FROM users", 1),
]

N_PER_CLASS = 600  # 600 + 600 = 1200 échantillons (> 1000 demandés)

rows = list(base_examples)
seen = set(q for q, _ in base_examples)

while len([r for r in rows if r[1] == 0]) < N_PER_CLASS:
    q = gen_legit()
    if q not in seen:
        rows.append((q, 0))
        seen.add(q)

while len([r for r in rows if r[1] == 1]) < N_PER_CLASS:
    q = gen_malicious()
    if q not in seen:
        rows.append((q, 1))
        seen.add(q)

random.shuffle(rows)

# Sauvegarde du dataset
os.makedirs("../dataset", exist_ok=True)
df = pd.DataFrame(rows, columns=["query", "label"])
df.to_csv("../dataset/sql_injection_dataset.csv", index=False)

print(f"Dataset généré et sauvegardé : {df.shape[0]} lignes")
print(df['label'].value_counts())


In [ ]:
# Chargement et aperçu du dataset
df = pd.read_csv("../dataset/sql_injection_dataset.csv")
print(f"Dimensions du dataset : {df.shape}")
df.head(10)


## Partie 3 — Prétraitement et Analyse Exploratoire

### 3.1 Nettoyage du texte

La normalisation des requêtes améliore la qualité de la vectorisation en réduisant la diversité inutile (majuscules/minuscules, espaces multiples).


In [ ]:
# ---------------------------------------------------------------
# Nettoyage du texte
# ---------------------------------------------------------------
def nettoyer_requete(query):
    """
    Nettoie une requête SQL :
    - conversion en minuscules
    - suppression des espaces multiples
    - suppression des espaces en début/fin de chaîne
    """
    query = str(query)
    query = query.lower()                     # conversion en minuscules
    query = re.sub(r'\s+', ' ', query)         # suppression des espaces multiples
    query = query.strip()                      # suppression des espaces inutiles
    return query


df['query_clean'] = df['query'].apply(nettoyer_requete)
df[['query', 'query_clean', 'label']].head(10)


### 3.2 Analyse exploratoire (EDA)


In [ ]:
# ---------------------------------------------------------------
# Analyse exploratoire du dataset (EDA)
# ---------------------------------------------------------------
print("Nombre total d'échantillons :", len(df))
print("\nRépartition des classes :")
print(df['label'].value_counts())
print("\nPourcentage par classe :")
print(df['label'].value_counts(normalize=True) * 100)

# Longueur des requêtes (nombre de caractères)
df['longueur'] = df['query_clean'].apply(len)
print("\nStatistiques sur la longueur des requêtes :")
print(df.groupby('label')['longueur'].describe())


In [ ]:
# ---------------------------------------------------------------
# Graphiques statistiques
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Répartition des classes
sns.countplot(data=df, x='label', hue='label', palette=['#2ecc71', '#e74c3c'],
              legend=False, ax=axes[0])
axes[0].set_title("Répartition des classes")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Légitime (0)", "Malveillante (1)"])
axes[0].set_xlabel("")
axes[0].set_ylabel("Nombre de requêtes")

# Distribution de la longueur des requêtes par classe
sns.boxplot(data=df, x='label', y='longueur', hue='label', palette=['#2ecc71', '#e74c3c'],
            legend=False, ax=axes[1])
axes[1].set_title("Longueur des requêtes par classe")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Légitime (0)", "Malveillante (1)"])
axes[1].set_xlabel("")
axes[1].set_ylabel("Longueur (caractères)")

plt.tight_layout()
plt.savefig("../dataset/eda_graphiques.png", dpi=120)
plt.show()


## Partie 4 — Feature Engineering : TF-IDF

### Pourquoi TF-IDF sur des n-grammes de caractères ?

Le `TfidfVectorizer` avec `analyzer='char_wb'` et `ngram_range=(1,3)` est particulièrement adapté à notre problème car :
- il capture des motifs courts et très discriminants (`'`, `--`, `or`, `1=1`, `union`) ;
- il est robuste aux variations d'espacement et de casse après normalisation ;
- il n'a pas besoin de tokenisation sémantique spécifique au SQL.


In [ ]:
# ---------------------------------------------------------------
# Feature Engineering : vectorisation TF-IDF
# ---------------------------------------------------------------
# On utilise un découpage par n-grammes de caractères (char_wb), qui est
# particulièrement adapté à la détection de motifs courts et spécifiques
# (apostrophes, commentaires SQL "--", mots-clés "OR", "UNION", etc.)
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(1, 3),
    max_features=3000
)

X = vectorizer.fit_transform(df['query_clean'])
y = df['label']

print("Taille du vocabulaire TF-IDF :", len(vectorizer.vocabulary_))
print("Dimensions de la matrice TF-IDF :", X.shape)


In [ ]:
# Mots/n-grammes les plus fréquents (somme des scores TF-IDF)
import numpy as np

scores = np.asarray(X.sum(axis=0)).ravel()
vocab = vectorizer.get_feature_names_out()
top_indices = scores.argsort()[::-1][:20]

top_features = pd.DataFrame({
    "ngramme": vocab[top_indices],
    "score_tfidf_total": scores[top_indices]
})
print("Top 20 des n-grammes les plus importants :")
top_features


In [ ]:
# ---------------------------------------------------------------
# Séparation train / test
# ---------------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['query_clean'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Taille de l'ensemble d'entraînement :", X_train.shape)
print("Taille de l'ensemble de test :", X_test.shape)


## Partie 5 — Modèles de Machine Learning

Quatre algorithmes de classification sont testés et comparés :

| Algorithme | Caractéristiques |
|---|---|
| **Logistic Regression** | Modèle linéaire rapide, bon baseline, interprétable |
| **Random Forest** | Ensemble d'arbres de décision, robuste, capte les non-linéarités |
| **SVM** | Séparateur à marge maximale, efficace en haute dimension |
| **Naive Bayes** | Basé sur les probabilités, très rapide, adapté aux fréquences TF-IDF |


In [ ]:
# ---------------------------------------------------------------
# Entraînement de plusieurs modèles de Machine Learning
# ---------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='linear', probability=True, random_state=42),
    "Naive Bayes": MultinomialNB(),
}

predictions = {}

for nom, modele in models.items():
    print(f"Entraînement du modèle : {nom} ...")
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    predictions[nom] = y_pred
    print(f"  -> Terminé.")

print("\nTous les modèles ont été entraînés avec succès.")


## Partie 6 — Évaluation des Modèles

Les métriques calculées pour chaque modèle :

| Métrique | Formule | Signification |
|---|---|---|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Taux global de bonnes classifications |
| **Precision** | TP/(TP+FP) | Parmi les alertes levées, quelle part est vraie ? |
| **Recall** | TP/(TP+FN) | Parmi les vraies injections, quelle part détecte-t-on ? |
| **F1 Score** | 2×(P×R)/(P+R) | Moyenne harmonique Précision/Rappel |


In [ ]:
# ---------------------------------------------------------------
# Partie 6 : Évaluation des modèles
# ---------------------------------------------------------------
resultats = {}

for nom, y_pred in predictions.items():
    resultats[nom] = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1 Score":  f1_score(y_test, y_pred),
    }

df_resultats = pd.DataFrame(resultats).T.round(4)
print("=== Résultats de l'évaluation des modèles ===")
print(df_resultats.to_string())


### Matrices de confusion


In [ ]:
# ---------------------------------------------------------------
# Matrices de confusion pour chaque modèle
# ---------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (nom, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=["Légitime", "Malveillante"],
                yticklabels=["Légitime", "Malveillante"])
    axes[i].set_title(f"Matrice de confusion — {nom}")
    axes[i].set_xlabel("Prédiction")
    axes[i].set_ylabel("Réalité")

plt.suptitle("Matrices de confusion — Comparaison des modèles", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../dataset/matrices_confusion.png", dpi=120, bbox_inches='tight')
plt.show()


### Courbes comparatives


In [ ]:
# ---------------------------------------------------------------
# Courbes comparatives des métriques
# ---------------------------------------------------------------
df_plot = df_resultats.reset_index().rename(columns={"index": "Modèle"})
df_melted = df_plot.melt(id_vars="Modèle", var_name="Métrique", value_name="Score")

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_melted, x="Modèle", y="Score", hue="Métrique",
                 palette="Set2")
ax.set_title("Comparaison des métriques par modèle", fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.12)
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.legend(loc="lower right")

# Affichage des valeurs
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)

plt.tight_layout()
plt.savefig("../dataset/courbes_comparatives.png", dpi=120)
plt.show()


## Partie 7 — Sélection du Meilleur Modèle


In [ ]:
# ---------------------------------------------------------------
# Partie 7 : Sélection du meilleur modèle
# ---------------------------------------------------------------
meilleur_nom = df_resultats["F1 Score"].idxmax()
meilleur_score = df_resultats.loc[meilleur_nom, "F1 Score"]

print(f"Meilleur modèle sélectionné : {meilleur_nom}")
print(f"F1 Score : {meilleur_score:.4f}")
print(f"\nRapport de classification détaillé :\n")
print(classification_report(y_test, predictions[meilleur_nom],
                             target_names=["Légitime (0)", "Malveillante (1)"]))

print(f"""
Justification du choix :
Le modèle "{meilleur_nom}" a obtenu le F1 Score le plus élevé ({meilleur_score:.4f}).
Le F1 Score est la métrique de référence pour les problèmes de classification binaire
car il équilibre Précision et Rappel, ce qui est crucial en cybersécurité :
- Maximiser le Rappel : ne pas laisser passer une vraie injection SQL (faux négatif).
- Maintenir une bonne Précision : ne pas bloquer trop de requêtes légitimes (faux positif).
""")


## Partie 8 — Sauvegarde du Modèle avec Joblib

Le modèle entraîné et le vectoriseur TF-IDF sont persistés sur disque pour être réutilisés dans l'interface Streamlit ou dans toute autre application.


In [ ]:
# ---------------------------------------------------------------
# Partie 8 : Sauvegarde du modèle entraîné
# ---------------------------------------------------------------
import os

os.makedirs("../models", exist_ok=True)

# Sauvegarde du meilleur modèle
chemin_modele = "../models/sql_injection_detector.pkl"
joblib.dump(models[meilleur_nom], chemin_modele)
print(f"Modèle sauvegardé : {chemin_modele}")

# Sauvegarde du vectoriseur TF-IDF (nécessaire pour les prédictions futures)
chemin_vectorizer = "../models/tfidf_vectorizer.pkl"
joblib.dump(vectorizer, chemin_vectorizer)
print(f"Vectoriseur TF-IDF sauvegardé : {chemin_vectorizer}")

# Vérification
taille = os.path.getsize(chemin_modele)
print(f"\nTaille du fichier .pkl : {taille / 1024:.1f} Ko")


## Partie 9 — Test en Temps Réel

La fonction `detect_sql_injection(query)` charge le modèle sauvegardé, nettoie et vectorise la requête passée en paramètre, puis retourne la classification.


In [ ]:
# ---------------------------------------------------------------
# Partie 9 : Test en temps réel — Fonction detect_sql_injection()
# ---------------------------------------------------------------

# Chargement du modèle et du vectoriseur sauvegardés
modele_charge = joblib.load("../models/sql_injection_detector.pkl")
vectorizer_charge = joblib.load("../models/tfidf_vectorizer.pkl")


def detect_sql_injection(query):
    """
    Analyse une requête SQL et détermine si elle est malveillante.

    Paramètres :
        query (str) : Texte de la requête SQL à analyser.

    Retour :
        str : "SQL Injection détectée" ou "Requête normale"
    """
    # Nettoyage de la requête
    query_clean = query.lower().strip()
    query_clean = re.sub(r'\s+', ' ', query_clean)

    # Vectorisation
    X_input = vectorizer_charge.transform([query_clean])

    # Prédiction
    prediction = modele_charge.predict(X_input)[0]

    if hasattr(modele_charge, "predict_proba"):
        proba = modele_charge.predict_proba(X_input)[0]
        confiance = proba[prediction]
        print(f"  Confiance : {confiance * 100:.1f}%")

    return "SQL Injection détectée" if prediction == 1 else "Requête normale"


# ---------------------------------------------------------------
# Tests en temps réel
# ---------------------------------------------------------------
requetes_test = [
    ("SELECT * FROM users WHERE username='admin' OR '1'='1'", "malveillante"),
    ("SELECT * FROM products WHERE id = 42",                  "légitime"),
    ("'; DROP TABLE users; --",                               "malveillante"),
    ("SELECT name, email FROM clients WHERE status='active'", "légitime"),
    ("' UNION SELECT username, password FROM users --",       "malveillante"),
    ("INSERT INTO orders (product_id, quantity) VALUES (5, 2)", "légitime"),
    ("' OR SLEEP(5) --",                                      "malveillante"),
    ("SELECT COUNT(*) FROM transactions",                     "légitime"),
]

print("=" * 60)
print("TEST EN TEMPS RÉEL — Détection SQL Injection")
print("=" * 60)
for requete, type_attendu in requetes_test:
    resultat = detect_sql_injection(requete)
    statut = "✓" if (
        (resultat == "SQL Injection détectée" and type_attendu == "malveillante") or
        (resultat == "Requête normale" and type_attendu == "légitime")
    ) else "✗"
    print(f"\n{statut} Requête  : {requete}")
    print(f"  Attendu  : {type_attendu}")
    print(f"  Résultat : {resultat}")
print("\n" + "=" * 60)


## Partie 10 — Conclusion


In [ ]:
# ---------------------------------------------------------------
# Partie 10 : Conclusion
# ---------------------------------------------------------------
print("""
╔══════════════════════════════════════════════════════════════╗
║                     CONCLUSION DU PROJET                    ║
╚══════════════════════════════════════════════════════════════╝

1. RÉSULTATS OBTENUS
   ─────────────────
   Le système de détection développé dans ce projet atteint des
   performances excellentes sur le dataset généré :

   • Meilleur modèle : {meilleur_nom}
   • F1 Score        : {meilleur_f1:.4f}
   • Accuracy        : {meilleur_acc:.4f}

   Tous les modèles testés (Logistic Regression, Random Forest,
   SVM, Naive Bayes) montrent une performance supérieure à 97 %,
   ce qui démontre l'efficacité du traitement TF-IDF à base de
   n-grammes de caractères pour la détection de motifs d'injection.

2. LIMITES DU SYSTÈME
   ───────────────────
   • Dataset synthétique : les requêtes ont été générées à partir
     de templates prédéfinis. Les performances pourraient varier
     sur des données réelles plus variées et bruitées.
   • Techniques d'évasion avancées : des attaquants expérimentés
     peuvent obfusquer leurs injections (encodage URL, commentaires
     imbriqués, Unicode, etc.) pour contourner des modèles simples.
   • Déséquilibre possible en production : les requêtes malveillantes
     sont rares en pratique ; un déséquilibre de classes peut
     affecter le taux de faux positifs/négatifs.
   • Pas de contexte applicatif : le modèle analyse chaque requête
     de manière isolée, sans tenir compte du comportement global
     d'un utilisateur ou d'une session.

3. PERSPECTIVES FUTURES
   ──────────────────────
   • Utilisation de datasets réels (logs de WAF, CSIC HTTP Dataset,
     dépôts OWASP) pour améliorer la généralisation du modèle.
   • Exploration de méthodes Deep Learning (LSTM, Transformers,
     BERT fine-tuné) pour capturer des dépendances séquentielles
     plus complexes dans les requêtes SQL.
   • Mise en place d'un système d'apprentissage en ligne (online
     learning) pour que le modèle s'adapte aux nouvelles techniques
     d'injection au fil du temps.
   • Intégration dans un WAF (Web Application Firewall) ou un
     proxy applicatif pour une protection en temps réel.
   • Enrichissement des features : taux de caractères spéciaux,
     présence de mots-clés SQL sensibles, longueur anormale, etc.
""".format(
    meilleur_nom=meilleur_nom,
    meilleur_f1=df_resultats.loc[meilleur_nom, "F1 Score"],
    meilleur_acc=df_resultats.loc[meilleur_nom, "Accuracy"],
))
